# Model v8: v4-C + consenso tra casi sovrapposti + calibrazione cross-unità
Cosa non sfruttano ancora v4-v7 (l'errore di v4-C è soprattutto di *regime settimanale*, non di forma dentro la settimana):
1. **Consenso temporale**: i casi sono giornalieri con orizzonte di 7 giorni, quindi ogni istante assoluto compare in 7 casi consecutivi (e nel test Kaggle i casi sono consecutivi). Si media la predizione dello stesso istante visto da casi diversi.
2. **Calibrazione per unità** (Platt su logit, monotona, calibrata sulle OOF): la micro AUC mescola tutte le unità, quindi conta anche l'allineamento tra unità, non solo l'AUC per unità.
3. **FUT (opzionale, da verificare col regolamento)**: volumi storici dei giorni *successivi* della settimana -> scarico realizzato per giorno. Dice quanta acqua è stata davvero usata, cioè il regime settimanale. Non sarebbe disponibile in una pianificazione reale.

In [10]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import joblib
from functools import cached_property
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view as swv
import yaml
import xgboost as xgb

In [11]:
# create the classes and funcs to prepare the data
def build_cases(X, Y, T=168):
    '''
    Modify the dataset to have them case indexed
    Input:
    - X: x dataset
    - Y: y dataset
    '''
    
    # splitting between daily and hourly features
    hourly = ['day_ahead_price'] + [c for c in X.columns if c.endswith('_inflow')]
    vol = [c for c in X.columns if c.endswith('_volume')]
    wv  = [c for c in X.columns if c.endswith('_wv')]

    # getting the position of every case starting hour
    pos = X.index.get_indexer(Y.index)
    assert (pos >= 0).all() and (pos + T < len(X)).all()

    # creating the sliding windows
    win = swv(X[hourly].to_numpy(np.float32), T, axis=0)       
    xh = win[pos].reshape(len(pos), -1)                        
    
    wv_arr = X[wv].to_numpy()
    xs = np.hstack([
        X[vol].to_numpy()[pos],  # initial volume (s)
        wv_arr[pos],             # initial water value (pos)
        wv_arr[pos + T]          # final water value (pos+T)
    ])

    # getting the names for the columns of x
    hourly_cols = [f"{c}_t{h}" for c in hourly for h in range(T)]
    wv_init_cols = [f"{c}_init" for c in wv]
    wv_final_cols = [f"{c}_final" for c in wv]
    
    all_cols = hourly_cols + vol + wv_init_cols + wv_final_cols

    # excluding run_no e total_calculation_time
    y = pd.DataFrame(
        Y.filter(like='result_committed').to_numpy(np.uint8),
        index=Y.index,
        columns=Y.columns[2:]
    )
    
    x = pd.DataFrame(
        np.hstack([xh, xs]).astype(np.float32),
        index=Y.index,
        columns=all_cols
    )
    
    return x, y


class DataLoader:
    # TODO: Change timestamp data from string to datatime object if available in the data source.
    # TODO: move this in the notebooks (or make it simpler/remove it)

    def __init__(self, data_source=None):
        self.data_source = Path(data_source) if data_source else data_dir

    def _csv(self, folder, filename, date_columns=()):
        file_path = self.data_source / folder / filename
        frame = pd.read_csv(file_path)
        for column in date_columns:
            if column in frame.columns:
                frame[column] = pd.to_datetime(frame[column])
        return frame

    def _yaml(self, folder, filename):
        file_path = self.data_source / folder / filename
        with file_path.open(encoding="utf-8") as stream:
            return yaml.safe_load(stream)

    @cached_property
    def day_ahead_price_data(self):
        return self._csv(
            "kernel",
            "Historical_day_ahead_price_2015_2025.csv",
            ("date",),
        )

    @cached_property
    def inflow_data(self):
        return self._csv("kernel", "Historical_inflow_1958_2025.csv", ("date",))

    @cached_property
    def volume_data(self):
        return self._csv("kernel", "Historical_volume_2015_2024.csv", ("date",))

    @cached_property
    def synthetic_water_value_data(self):
        return self._csv(
            "kernel",
            "Synthetic_water_value_2015_2024.csv",
            ("date",),
        )

    @cached_property
    def commitment_descision_data(self):
        return self._csv("kernel", "Unit_commitment_decisions.csv", ("starttime",))

    @cached_property
    def commitment_decision_data(self):
        return self.commitment_descision_data

    @cached_property
    def min_flow_constraint_data(self):
        return self._csv("extended", "Constraint_min_flow.csv", ("date",))

    @cached_property
    def min_volume_constraint_data(self):
        return self._csv("extended", "Constraint_min_volume.csv", ("date",))

    @cached_property
    def topology_data(self): #TODO: CHECK THIS ONE LATER
        return self._yaml("extended", "Tokke_Vinje_topology.yaml")



# Class to create the entire dataset in one go
class Training_set(DataLoader):
    '''
    It imports all the dataset, merging them using the the index (that will be the date)
    '''
 
    def __init__(self, data_source=None):
        super().__init__(data_source)
        self.y = None
        self.x = None
 
    def load_copies(self):
        sources = {
            "day_ahead_price": self.day_ahead_price_data,
            "inflow": self.inflow_data,
            "volume": self.volume_data,
            "wv": self.synthetic_water_value_data,
            "uc": self.commitment_decision_data,
        }
        return sources
 
 
    def create_x(self):
        '''
        Create the complete X dataset
        
        TODO: feature engineering and selection
                + time split for easier computation (test model on smaller data)
        '''
        
        data = self.load_copies()
        
        # setting the date as index and fixing the columns names
        for name in data.keys():
            if 'date' in list(data[name].columns):
                data[name]['date'] = pd.to_datetime(data[name]['date'], utc=True)
                data[name] = data[name].set_index(data[name]['date']).drop(columns='date')
                if 'Day-ahead price (EUR/MWh)' in data[name].columns:
                    data[name].columns = data[name].columns.str.replace(r'\s*\([^)]*\)', '', regex=True).str.replace(r'[\s-]+', '_', regex=True).str.lower()
                else:
                    data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')
                
            else:
                data[name]['starttime'] = pd.to_datetime(data[name]['starttime'], utc=True)
                data[name] = data[name].set_index(data[name]['starttime']).drop(columns='starttime')
                data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')

        # wv and volume have daily data
        #   need to repet thier value hourly
        hourly = data['day_ahead_price'].index
        for name in ('volume', 'wv'):
            data[name] = data[name].reindex(hourly, method='ffill')
        
        # creating the two datset
        #   y stored for future use
        self.y = data['uc'].copy()
        data.pop('uc')
        x = data['day_ahead_price']
        data.pop('day_ahead_price')
        for name in data.keys():
                x = x.join(data[name].add_suffix('_' + name), how='left')
                
        
        self.x = x.copy()
    
    
    def split_small_data(self, begin_train:str ='2015-01-01', end_train: str='2018-12-31', begin_test: str='2019-01-12', end_test: str='2019-12-31'):
        '''
        Return X and Y dataset
        
        Input:
            - beginnig_train (str, dafault=2015-01-01 ): strart date training, format 'yyyy-mm-dd'
            - beginnig_test (str, dafault=2018-12-31 ): strart date testing, format 'yyyy-mm-dd'
            - end_train (str, dafault=2019-01-01 ): end date of training, format 'yyyy-mm-dd'
            - end_test (str, dafault=2019-12-31 ): end date of testing, format 'yyyy-mm-dd'
        '''
        
        self.create_x()
        X, Y = self.x.copy(), self.y.copy()
        x_train, y_train = build_cases(X, Y.loc[begin_train:end_train])
        x_test,  y_test  = build_cases(X, Y.loc[begin_test:end_test])
        return x_train, y_train, x_test, y_test

In [ ]:
DATA = './Users/lucaag/data'
ts = Training_set(data_source=DATA)
ts.create_x()
X_all, Y_all = ts.x.sort_index(), ts.y.sort_index()

split = int(len(Y_all) * 0.8)  # stessa split di v4
x_train, y_train = build_cases(X_all, Y_all.iloc[:split])
x_test, y_test = build_cases(X_all, Y_all.iloc[split + 7:])
topo = ts.topology_data['model']
for x in (x_train, x_test): assert (x.index.to_series().diff().dropna() == pd.Timedelta('1D')).all()  # casi consecutivi
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(2337, 3243) (2337, 2352) (578, 3243) (578, 2352)


## Topologia e target (come v4)

In [13]:
T = 168
RES = {k.lower(): v for k, v in topo['reservoir'].items()}
PL = {k.lower().replace(' ', '_'): v for k, v in topo['plant'].items()}
TOPO = {
    'vesle_kjela': (['kjelavatn'], 'foersvatn'),
    'kjela': (['foersvatn', 'bordalsvatn'], 'hyljelihyl'),
    'songa': (['songavatn', 'bitdalsvatn'], 'totak'),
    'vinje': (['vaamarvatn', 'totak'], 'vinjevatn'),
    'haukeli': (['vatjern', 'langeidvatn'], 'vinjevatn'),
    'tokke': (['vinjevatn'], 'bandak'),
    'byrte': (['botnedalsvatn'], 'byrtevatn'),
    'lio': (['byrtevatn'], 'bandak'),
    'hogga': (['bandak'], None),
}
UP_PLANTS = {'kjela': ['vesle_kjela'], 'vinje': ['kjela', 'songa'], 'tokke': ['vinje', 'haukeli'],
             'hogga': ['tokke', 'lio'], 'lio': ['byrte']}
ORDER = ['vesle_kjela', 'songa', 'byrte', 'haukeli', 'kjela', 'lio', 'vinje', 'tokke', 'hogga']

units = sorted({c[len('result_committed_'):].rsplit('_t', 1)[0] for c in y_train.columns})
plant_units = {p: [i for i, u in enumerate(units) if u.rsplit('_g', 1)[0] == p] for p in ORDER}
ucols = lambda u: [f'result_committed_{units[u]}_t{h}' for h in range(T)]
stack3 = lambda y: np.stack([y[ucols(u)].to_numpy() for u in range(len(units))], axis=1)  # (casi, unità, 168)
Y3, Y3te = stack3(y_train), stack3(y_test)
Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]
out_cols = [c for u in range(len(units)) for c in ucols(u)]

## Diagnosi: quanto sono coerenti i casi consecutivi?
Accordo tra (caso d, ora t+24) e (caso d+1, ora t) sulle label vere: se è alto, la media tra viste riduce davvero il rumore.

In [14]:
agree = (Y3[:-1, :, 24:] == Y3[1:, :, :-24]).mean((0, 2))
pd.Series(agree, units).round(3)

byrte_g1          0.964
haukeli_g1        0.869
hogga_g1          0.843
kjela_g1          0.942
lio_g1            0.959
songa_g1          0.950
tokke_g1          0.847
tokke_g2          0.811
tokke_g3          0.822
tokke_g4          0.814
vesle_kjela_g1    0.960
vinje_g1          0.821
vinje_g2          0.812
vinje_g3          0.823
dtype: float64

## Feature (v4 + FUT opzionale)

In [15]:
PRICE = [f'day_ahead_price_t{h}' for h in range(T)]

def shift(a, k):
    o = np.roll(a, k, 1)
    if k > 0: o[:, :k] = a[:, :1]
    else: o[:, k:] = a[:, -1:]
    return o

def head(r, v):
    d = RES[r]['vol_head']
    return np.interp(v, d['x'], d['y'])

def static(X, plant):
    up, dn = TOPO[plant]; rs = up + ([dn] if dn else [])
    S = [X[[f'{r}_{s}' for r in rs for s in ('volume', 'wv_init', 'wv_final')]].to_numpy()]
    S += [(X[f'{r}_volume'] / RES[r]['max_vol']).to_numpy()[:, None] for r in rs]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else np.zeros_like(hu)
    wd = X[f'{dn}_wv_final'].to_numpy() if dn else np.zeros(len(X))
    S.append(np.column_stack([hu, hu - np.maximum(hd, PL[plant]['outlet_line']),
                              X[f'{up[0]}_wv_final'].to_numpy() - wd]))
    return np.hstack(S)

def inflow(X, up):  # inflow orario sommato sui bacini a monte (m3/s)
    return sum((X[[f'{r}_inflow_t{h}' for h in range(T)]].to_numpy() for r in up if f'{r}_inflow_t0' in X.columns),
               np.zeros((len(X), T), np.float32))

def fut_feats(X, plant):  # ATTENZIONE: usa i volumi storici dei giorni FUTURI della settimana
    up, n = TOPO[plant][0], len(X)
    pos = X_all.index.get_indexer(X.index)[:, None] + 24 * np.arange(8)
    V = sum(X_all[f'{r}_volume'].to_numpy()[pos] for r in up)  # (n, 8) Mm3 a inizio giorno
    D = inflow(X, up).reshape(n, 7, 24).sum(2) * 0.0036 - np.diff(V, axis=1)  # scarico stimato per giorno (Mm3)
    return [np.repeat(D, 24, 1), np.repeat(D.sum(1, keepdims=True), T, 1), np.repeat(V[:, -1:] - V[:, :1], T, 1)]

def base(X, plant, up_prob=None, fut=False):  # matrice (n*168, F)
    up, n = TOPO[plant][0], len(X)
    P = X[PRICE].to_numpy(); D = P.reshape(n, 7, 24)
    I = inflow(X, up)
    wvf = X[f'{up[0]}_wv_final'].to_numpy()[:, None]
    h = np.tile(np.arange(T), (n, 1))
    F = [h, h % 24, (X.index.dayofweek.to_numpy()[:, None] + h // 24) % 7,
         P, shift(P, 1), shift(P, -1), P.argsort(1).argsort(1) / (T - 1),
         D.argsort(2).argsort(2).reshape(n, T) / 23, P - np.repeat(D.mean(2), 24, 1),
         P - P.mean(1, keepdims=True), P - wvf, I, np.repeat(I.mean(1, keepdims=True), T, 1)]
    if up_prob is not None:
        F += [up_prob, shift(up_prob, 1), shift(up_prob, 3), np.repeat(up_prob.mean(1, keepdims=True), T, 1)]
    if fut: F += fut_feats(X, plant)
    A = np.column_stack([f.ravel() for f in F] + [np.repeat(static(X, plant), T, 0),
                                                  np.repeat(X.index.dayofyear.to_numpy(), T)])
    return A.astype(np.float32)

## Training: modello per centrale, OOF a blocchi contigui per *tutte* le centrali (servono per la calibrazione)

In [16]:
NE, FOLDS, GAP, SEEDS = 200, 5, 7, 1  # SEEDS=3 per la submission finale

def mk(seed):
    return xgb.XGBClassifier(n_estimators=NE, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             tree_method='hist', n_jobs=-1, random_state=seed)

def fp(X, y, Xs):
    if y.min() == y.max():
        return [np.full(len(x), float(np.clip(y[0], .01, .99))) for x in Xs]
    ms = [mk(s).fit(X, y) for s in range(SEEDS)]
    return [np.mean([m.predict_proba(x)[:, 1] for m in ms], 0) for x in Xs]

def folds(n):
    b = np.linspace(0, n, FOLDS + 1, dtype=int)
    for a, c in zip(b[:-1], b[1:]):
        yield np.r_[0:max(a - GAP, 0), min(c + GAP, n):n], np.arange(a, c)

ridx = lambda c: (np.asarray(c)[:, None] * T + np.arange(T)).ravel()
design = lambda B, c, us: np.vstack([np.column_stack([B[ridx(c)], np.full(len(ridx(c)), u, np.float32)]) for u in us])
ylong = lambda c, us: np.concatenate([Yflat[u][ridx(c)] for u in us])

def fit_group(B, Bt, us):
    n, nt = len(B) // T, len(Bt) // T
    allc = np.arange(n)
    te = fp(design(B, allc, us), ylong(allc, us), [design(Bt, np.arange(nt), us)])[0].reshape(len(us), nt, T)
    oo = np.zeros((len(us), n, T))
    for tr, va in folds(n):
        p = fp(design(B, tr, us), ylong(tr, us), [design(B, va, us)])[0]
        oo[:, va] = p.reshape(len(us), len(va), T)
    return oo, te

def run(fut=False):  # ritorna predizioni test e OOF, entrambe (casi, unità, 168)
    oof, te = {}, {}
    P = np.zeros((len(x_test), len(units), T), np.float32); O = np.zeros((len(x_train), len(units), T), np.float32)
    for plant in ORDER:
        us, ups = plant_units[plant], UP_PLANTS.get(plant, [])
        up_tr = np.mean([oof[p] for p in ups], 0) if ups else None
        up_te = np.mean([te[p] for p in ups], 0) if ups else None
        oo, t = fit_group(base(x_train, plant, up_tr, fut), base(x_test, plant, up_te, fut), us)
        P[:, us, :], O[:, us, :] = t.transpose(1, 0, 2), oo.transpose(1, 0, 2)
        te[plant], oof[plant] = t.mean(0), oo.mean(0)
        print(plant, 'ok')
    return P, O

## Post-processing: consenso tra casi sovrapposti e calibrazione per unità

In [17]:
def consensus(P, b, K=6):
    # P (casi consecutivi, unità, 168). Stesso istante assoluto: caso d+k, ora t-24k. Peso 1 al caso stesso, b a ogni vista
    n = len(P); S = np.zeros_like(P); C = np.zeros((n, 1, T), np.float32)
    for k in range(1, K + 1):
        w = 24 * k
        S[:n-k, :, w:] += P[k:, :, :T-w];  C[:n-k, :, w:] += 1   # viste dai casi successivi
        S[k:, :, :T-w] += P[:n-k, :, w:];  C[k:, :, :T-w] += 1   # viste dai casi precedenti
    return (P + b * S) / (1 + b * C)

logit = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4))).reshape(-1, 1)

def calib(Poof, Yoof, Pte):  # Platt per unità sulle OOF: monotona, cambia solo l'allineamento tra unità
    out = np.array(Pte, dtype=np.float32)
    for u in range(Pte.shape[1]):
        y = Yoof[:, u].ravel()
        if y.min() == y.max(): continue
        m = LogisticRegression(C=100).fit(logit(Poof[:, u]), y)
        out[:, u] = m.predict_proba(logit(Pte[:, u]))[:, 1].reshape(Pte[:, u].shape)
    return out

auc = lambda Y, P: roc_auc_score(Y.ravel(), P.ravel())  # micro AUC

## Confronto (v4-C su questa split: 0.9423). Scegliere b su `AUC oof`, non su test

In [ ]:
table, preds = [], {}
for name, fut in {'C (v4)': False, 'C + FUT': True}.items():
    P, O = run(fut); preds[name] = (P, O)
    for b in (0, .25, .5, 1):
        Pc, Oc = consensus(P, b), consensus(O, b)
        table.append((name, b, False, auc(Y3, Oc), auc(Y3te, Pc)))
        table.append((name, b, True, np.nan, auc(Y3te, calib(Oc, Y3, Pc))))
pd.DataFrame(table, columns=['modello', 'b', 'calib', 'AUC oof', 'AUC test'])

vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok


,modello,b,calib,AUC oof,AUC test
0,C (v4),0.00,False,0.961720,0.942348
1,C (v4),0.00,True,NaN,0.935826
2,C (v4),0.25,False,0.941341,0.930161
3,C (v4),0.25,True,NaN,0.922363
4,C (v4),0.50,False,0.928223,0.918453
5,C (v4),0.50,True,NaN,0.911526
6,C (v4),1.00,False,0.915738,0.906831
7,C (v4),1.00,True,NaN,0.900723
8,C + FUT,0.00,False,0.962543,0.944513
9,C + FUT,0.00,True,NaN,0.937834


## File di submission (dopo aver scelto modello e b)

In [ ]:
def to_df(P, index, columns):
    return pd.DataFrame(P.reshape(len(P), -1), index=index, columns=out_cols)[columns]
# P, O = preds['C (v4)']; sub = to_df(calib(consensus(O, b), Y3, consensus(P, b)), y_test.index, y_test.columns)